# Task 3: Music Generation with AI
A deep learning model trained on MIDI music data to generate new music sequences.
Note sequences are extracted from MIDI files using music21, fed into an LSTM neural network,
and the generated output is saved as a new MIDI file.

In [11]:
# Installing required libraries
!pip install music21
!pip install requests tqdm
!pip install music21 requests tqdm tensorflow numpy

  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl (437 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.34.1 which is incompatible.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [14]:
# Download Chopin MIDI files for training
import urllib.request
import os

# Create directory to store MIDI files
os.makedirs("midi_files", exist_ok=True)

# Verify MIDI files are accessible
import os

midi_files = [f for f in os.listdir("midi_files") if f.endswith(".mid") or f.endswith(".midi")]
print(f"Total MIDI files found: {len(midi_files)}")
for f in midi_files:
    print(f" - {f}")

Total MIDI files found: 5
 - chpn_op10_e01.mid
 - chpn_op7_1.mid
 - chpn_op7_2.mid
 - chp_op18.mid
 - chp_op31.mid


In [16]:
# Imports for MIDI parsing and note extraction
from music21 import converter, instrument, note, chord
import glob

def extract_notes(midi_folder):
    notes = []
    
    # Parse each MIDI file and extract notes and chords
    for file in glob.glob(f"{midi_folder}/*.mid"):
        print(f"Processing: {file}")
        midi = converter.parse(file)
        
        # Directly iterate over all notes and chords in the flattened stream
        for element in midi.flat.notes:
            if isinstance(element, note.Note):
                # Single note — store pitch as string
                notes.append(str(element.pitch))
            elif isinstance(element, chord.Chord):
                # Chord — store as dot-separated note values
                notes.append('.'.join(str(n) for n in element.normalOrder))
    
    return notes

# Extract all notes from the dataset
notes = extract_notes("midi_files")
print(f"\nTotal notes extracted: {len(notes)}")
print(f"Sample notes: {notes[:10]}")
print(f"Unique notes: {len(set(notes))}")

Processing: midi_files\chpn_op10_e01.mid


D:\anaconda3\Lib\site-packages\music21\stream\base.py:3675: Music21DeprecationWarning: .flat is deprecated.  Call .flatten() instead
  return self.iter().getElementsByClass(classFilterList)


Processing: midi_files\chpn_op7_1.mid
Processing: midi_files\chpn_op7_2.mid
Processing: midi_files\chp_op18.mid
Processing: midi_files\chp_op31.mid

Total notes extracted: 10315
Sample notes: ['C2', 'C3', 'C3', 'G3', 'C4', 'E4', 'C4', 'G4', 'C5', 'E5']
Unique notes: 216


In [17]:
# Imports for data preparation
import numpy as np
from tensorflow.keras.utils import to_categorical

# Sequence length: how many notes the model sees before predicting the next one
SEQUENCE_LENGTH = 50

# Map unique notes to integers
unique_notes = sorted(set(notes))
note_to_int = {note: number for number, note in enumerate(unique_notes)}
int_to_note = {number: note for number, note in enumerate(unique_notes)}
n_vocab = len(unique_notes)

# Build input/output sequence pairs
input_sequences = []
output_notes = []

for i in range(len(notes) - SEQUENCE_LENGTH):
    input_seq = notes[i:i + SEQUENCE_LENGTH]
    output_note = notes[i + SEQUENCE_LENGTH]
    input_sequences.append([note_to_int[n] for n in input_seq])
    output_notes.append(note_to_int[output_note])

# Reshape and normalize input for LSTM
X = np.reshape(input_sequences, (len(input_sequences), SEQUENCE_LENGTH, 1))
X = X / float(n_vocab)

# One-hot encode output
y = to_categorical(output_notes, num_classes=n_vocab)

print(f"Training samples: {len(X)}")
print(f"Vocabulary size: {n_vocab}")
print(f"Input shape: {X.shape}")
print(f"Output shape: {y.shape}")

Training samples: 10265
Vocabulary size: 216
Input shape: (10265, 50, 1)
Output shape: (10265, 216)


In [18]:
# Imports for model building
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint

# Build LSTM model
model = Sequential([
    LSTM(128, input_shape=(X.shape[1], X.shape[2]), return_sequences=True),
    Dropout(0.3),
    LSTM(128),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(n_vocab, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam')
model.summary()

# Save best model weights during training
checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor='loss',
    save_best_only=True,
    mode='min'
)

# Train the model: kept to 10 epochs for CPU
print("\nStarting training...")
history = model.fit(X, y, epochs=10, batch_size=64, callbacks=[checkpoint])
print("Training complete.")

D:\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50, 128)        │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 216)            │        14,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 220,440 (861.09 KB)

 Trainable params: 220,440 (861.09 KB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 51s 227ms/step - loss: 4.8160
Epoch 2/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 35s 218ms/step - loss: 4.7129
Epoch 3/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 36s 224ms/step - loss: 4.7090
Epoch 4/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 35s 215ms/step - loss: 4.7043
Epoch 5/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 36s 220ms/step - loss: 4.6741
Epoch 6/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 36s 224ms/step - loss: 4.5696
Epoch 7/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 39s 242ms/step - loss: 4.5014
Epoch 8/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 38s 234ms/step - loss: 4.4502
Epoch 9/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 35s 215ms/step - loss: 4.4144
Epoch 10/10
161/161 ━━━━━━━━━━━━━━━━━━━━ 43s 225ms/step - loss: 4.3796
Training complete.


In [19]:
# Imports for music generation
from tensorflow.keras.models import load_model
import random

# Load best saved model weights
model.load_weights("best_model.keras")

# Generate notes from a random seed sequence
def generate_notes(model, input_sequences, int_to_note, n_vocab, num_notes=100):
    # Pick a random starting sequence from training data
    start = random.randint(0, len(input_sequences) - 1)
    pattern = input_sequences[start]
    generated_notes = []

    print("Generating notes...")
    for _ in range(num_notes):
        # Reshape and normalize input
        input_array = np.reshape(pattern, (1, len(pattern), 1))
        input_array = input_array / float(n_vocab)

        # Predict next note
        prediction = model.predict(input_array, verbose=0)
        index = np.argmax(prediction)
        generated_notes.append(int_to_note[index])

        # Slide the window forward
        pattern = pattern[1:]
        pattern.append(index)

    return generated_notes

# Generate 100 notes
generated_notes = generate_notes(model, input_sequences, int_to_note, n_vocab, num_notes=100)
print(f"Generated {len(generated_notes)} notes")
print(f"Sample: {generated_notes[:20]}")

Generating notes...
Generated 100 notes
Sample: ['8.0.3', '8.0.3', '8.0.3', '8.0.3', '8.0.3', '8.0.3', '8.0.3', '8.0.3', '8.0.3', 'G#4', '8.0.3', '8.0.3', '8.0.3', '8.0.3', '8.0.3', 'F5', 'F5', 'F5', 'F5', 'F5']


In [20]:
# Imports for MIDI conversion and saving
from music21 import stream, note, chord
from fractions import Fraction

def notes_to_midi(generated_notes, output_file="output.mid"):
    output_stream = stream.Stream()
    offset = 0

    for pattern in generated_notes:
        # If pattern contains dots it is a chord
        if '.' in pattern:
            chord_notes = []
            for n in pattern.split('.'):
                try:
                    new_note = note.Note(int(float(n)))
                    new_note.storedInstrument = None
                    chord_notes.append(new_note)
                except:
                    pass
            if chord_notes:
                new_chord = chord.Chord(chord_notes)
                new_chord.offset = offset
                output_stream.append(new_chord)
        else:
            # Single note
            try:
                new_note = note.Note(pattern)
                new_note.offset = offset
                output_stream.append(new_note)
            except:
                pass
        offset += 0.5

    output_stream.write('midi', fp=output_file)
    print(f"MIDI saved as: {output_file}")

# Convert and save generated notes to MIDI
notes_to_midi(generated_notes, output_file="output.mid")

MIDI saved as: output.mid


In [21]:
# Play the generated MIDI file using music21
from music21 import converter

generated_midi = converter.parse("output.mid")
generated_midi.show('midi')